# Train V3 Corrected Structural Mapping: HC3 → GPT-3.5 Mixed (0% noise), batch 64

Notebook này train **V3 Corrected Structural Mapping** (`structmap`) trên HC3 train/validation, chọn checkpoint chỉ theo HC3 validation, rồi test **duy nhất** GPT-3.5 Mixed clean (0% noise). Đây là biến thể V3 riêng, không phải baseline legacy `baseline`; nó dùng `CorrectedStructuralMappingNet` và bắt buộc `--struct-mode directed`.

`--batch-size 64` là batch tối ưu hiệu dụng. `--micro-batch-size 8` chỉ giới hạn donor pool để vừa GPU. Không xây dựng hoặc đọc graph test trước khi train.

In [ ]:
# Kaggle Python 3.13 needs current binary wheels. Do not pin transformers==4.44.2.
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
import torch_geometric
import transformers
import spacy
print({
    'torch': torch.__version__,
    'torch_geometric': torch_geometric.__version__,
    'transformers': transformers.__version__,
    'spacy': spacy.__version__,
})

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

WORK = Path('/kaggle/working/group_v3_structmap_hc3_to_gpt35_bs64')
REPO = WORK / 'PRDetect-Research'
BRANCH = 'feat/rgcn-rdrop'
SEED = 2026
VARIANT = 'structmap'
STRUCT_MODE = 'directed'
BATCH_SIZE = 64
MICRO_BATCH_SIZE = 8
EPOCHS = 10
TEST_SPLIT = 'gpt3.5_mixed_test_split'
MODEL_DIR = WORK / 'model'
EVAL_DIR = WORK / 'evaluation'
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                'https://github.com/tphan1420/PRDetect-Research.git', str(REPO)], check=True)
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
assert (REPO / 'train_v3.py').is_file() and (REPO / 'evaluate_v3_quang.py').is_file()
assert (REPO / 'audit_correctedstruct.py').is_file()
assert (REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json').is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph must not exist before training.'
print({'commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip(),
       'variant': VARIANT, 'struct_mode': STRUCT_MODE, 'seed': SEED,
       'effective_batch_size': BATCH_SIZE, 'micro_batch_size': MICRO_BATCH_SIZE,
       'test_only': TEST_SPLIT})

In [ ]:
# Build only HC3 training/validation graphs. The GPT-3.5 test graph is deliberately excluded.
subprocess.run([sys.executable, 'building_graph.py', '--file', 'hc3_train', 'hc3_val'], cwd=REPO, check=True)
TRAIN_GRAPH = REPO / 'graph_data/hc3_train.pkl'
VAL_GRAPH = REPO / 'graph_data/hc3_val.pkl'
assert TRAIN_GRAPH.is_file() and VAL_GRAPH.is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph was created before training.'

In [ ]:
subprocess.run(
    [sys.executable, 'train_v3.py', '--variants', VARIANT, '--seeds', str(SEED),
     '--graph-dir', str(REPO / 'graph_data'), '--output-dir', str(MODEL_DIR),
     '--train-file', 'hc3_train', '--val-file', 'hc3_val', '--struct-mode', STRUCT_MODE,
     '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
     '--val-batch-size', '64', '--epochs', str(EPOCHS), '--patience', '10', '--device', 'cuda'],
    cwd=REPO,
    check=True,
)
CHECKPOINT = MODEL_DIR / f'v3_{VARIANT}_seed{SEED}.pt'
TRAIN_METADATA = CHECKPOINT.with_suffix('.train.json')
assert CHECKPOINT.is_file() and TRAIN_METADATA.is_file()

In [ ]:
# Build and evaluate exactly one post-training test split: GPT-3.5 Mixed 0%.
subprocess.run([sys.executable, 'building_graph.py', '--file', TEST_SPLIT], cwd=REPO, check=True)
GRAPH = REPO / 'graph_data' / f'{TEST_SPLIT}.pkl'
assert GRAPH.is_file()

from types import SimpleNamespace
import importlib.util
spec = importlib.util.spec_from_file_location('v3_eval', REPO / 'evaluate_v3_quang.py')
v3_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v3_eval)
args = SimpleNamespace(variant=VARIANT, train_domain='hc3', checkpoint_dir=str(MODEL_DIR), seeds=[SEED],
                       batch_size=BATCH_SIZE, struct_mode=STRUCT_MODE, device='cuda', graph_budget_root=str(WORK))
config = v3_eval.make_config(args)
CONFIG = EVAL_DIR / 'config.json'
v3_eval.write_json(CONFIG, config)
test_source = REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json'
expected_samples = sum(1 for line in test_source.open(encoding='utf-8') if line.strip())
assert expected_samples == 1000
REPORT = EVAL_DIR / 'gpt35_0pct_report.json'
subprocess.run([sys.executable, 'evaluate_v3_quang.py', '_test', '--config', str(CONFIG), '--seed', str(SEED),
                '--graph', str(GRAPH), '--expected-samples', str(expected_samples),
                '--signature', 'single_gpt35_0pct_after_hc3_structmap_training', '--report', str(REPORT)],
               cwd=REPO, check=True)
assert REPORT.is_file()

In [ ]:
# Audit provenance and directed-cache invariants; it reads artifacts only and refuses to overwrite a report.
AUDIT = EVAL_DIR / 'correctedstruct_audit.json'
subprocess.run([sys.executable, 'audit_correctedstruct.py', '--checkpoint-dir', str(MODEL_DIR),
                '--seeds', str(SEED), '--graph-cache', str(TRAIN_GRAPH), str(VAL_GRAPH), str(GRAPH),
                '--output', str(AUDIT)], cwd=REPO, check=True)
assert AUDIT.is_file()

import zipfile
report = json.loads(REPORT.read_text(encoding='utf-8'))
archive = WORK / 'v3_corrected_structmap_hc3_to_gpt35_0pct_batch64_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as out:
    for path in (CHECKPOINT, TRAIN_METADATA, CONFIG, REPORT, AUDIT):
        out.write(path, path.name)
print(json.dumps({key: report[key] for key in ('samples', 'accuracy', 'auc', 'f1', 'recall_ai', 'recall_human', 'threshold', 'validation_threshold')}, indent=2))
print('ZIP:', archive)